In [ ]:
# ==============================================================================
# TAHAP 1: INSTALASI PUSTAKA DAN PERSIAPAN LINGKUNGAN KERJA
# ==============================================================================
# Menginstal pustaka rasterio (parameter -q digunakan agar tampilan instalasi lebih ringkas)
!pip install rasterio -q

# Menghubungkan Google Colab dengan Google Drive pengguna
from google.colab import drive
drive.mount('/content/drive')

print("\nTahap 1 Selesai: Lingkungan kerja siap.\n")

Mounted at /content/drive

Tahap 1 Selesai: Lingkungan kerja siap.



In [ ]:
# ==============================================================================
# TAHAP 2: IMPOR MODUL DAN KONFIGURASI DIREKTORI
# ==============================================================================
import os
import glob
import re
import numpy as np
import rasterio
import warnings

# Definisikan path utama berdasarkan struktur direktori Google Drive
base_path = '/content/drive/MyDrive/Colab Notebooks - jariyanarifudin@mail.ugm.ac.id/Skripsi'

# Folder utama untuk menyimpan hasil akhir agregasi statistik periode 2010-2020
output_folder = os.path.join(base_path, 'Statistik_SPI_Output_2010_2020')
os.makedirs(output_folder, exist_ok=True)

# Pemetaan konfigurasi letak folder input dan pola nama file untuk tiap skala SPI
spi_configs = {
    'SPI-1': {
        'input_dir': os.path.join(base_path, 'SPI-Output'),
        'pattern': 'SPI-1_*.tif'
    },
    'SPI-3': {
        'input_dir': os.path.join(base_path, 'SPI3-Output'),
        'pattern': 'SPI3_*.tif'
    },
    'SPI-6': {
        'input_dir': os.path.join(base_path, 'SPI6-Output'),
        'pattern': 'SPI6_*.tif'
    },
    'SPI-12': {
        'input_dir': os.path.join(base_path, 'SPI12-Output'),
        'pattern': 'SPI12_*.tif'
    }
}

In [ ]:
# ==============================================================================
# TAHAP 3: KOMPUTASI BERBASIS BLOK DENGAN FILTER UNIK (JAN 2010 - DES 2020)
# ==============================================================================
for spi_name, config in spi_configs.items():
    print(f"{'='*60}")
    print(f"Memulai agregasi spasial untuk {spi_name}...")

    # Pencarian seluruh file .tif secara rekursif di dalam direktori
    all_files = glob.glob(os.path.join(config['input_dir'], '**', config['pattern']), recursive=True)

    # Menggunakan struktur Dictionary untuk menghilangkan duplikasi
    unique_files = {}

    for f in all_files:
        file_name = os.path.basename(f)

        # Ekstrak identitas waktu YYYYMM
        match = re.search(r'(\d{6})', file_name)
        if match:
            date_str = match.group(1)

            # Filter ketat untuk rentang studi (Januari 2010 - Desember 2020)
            if '201001' <= date_str <= '202012':
                # Jika file dengan bulan yang sama ditemukan lagi di sub-folder lain,
                # ia akan menimpa (overwrite) yang lama, memastikan hanya ada 1 file per bulan
                unique_files[date_str] = f

    # Ekstrak path file yang sudah terjamin unik dan urutkan secara kronologis
    raster_files = sorted(unique_files.values())

    if not raster_files:
        print(f"Peringatan: Tidak ada file rentang 2010-2020 yang ditemukan untuk {spi_name}")
        continue

    print(f"Ditemukan {len(raster_files)} file raster UNIK dalam rentang Jan 2010 - Des 2020.")

    # Pendefinisian nama file hasil keluaran
    output_mean = os.path.join(output_folder, f'{spi_name}_mean_2010_2020.tif')
    output_median = os.path.join(output_folder, f'{spi_name}_median_2010_2020.tif')

    # Ekstraksi metadata dari file pertama sebagai parameter georeferensi dasar
    with rasterio.open(raster_files[0]) as src_ref:
        meta = src_ref.meta.copy()

    # Penyesuaian tipe data ke float32 untuk pengolahan desimal dan akomodasi nilai NaN
    meta.update(dtype=rasterio.float32, nodata=np.nan)

    # Membuka koneksi data raster input terpilih secara bersamaan
    src_files = [rasterio.open(f) for f in raster_files]

    try:
        with rasterio.open(output_mean, 'w', **meta) as dst_mean, \
             rasterio.open(output_median, 'w', **meta) as dst_median:

            # Iterasi pemrosesan berbasis jendela spasial (window) untuk efisiensi RAM
            for ji, window in src_files[0].block_windows(1):
                window_arrays = []

                for src in src_files:
                    data_blok = src.read(1, window=window).astype('float32')
                    # Mengubah nilai NoData bawaan menjadi NaN agar komputasi statistik valid
                    if src.nodata is not None:
                        data_blok[data_blok == src.nodata] = np.nan
                    window_arrays.append(data_blok)

                # Transformasi data blok menjadi tumpukan matriks (stack) 3 dimensi
                stack_blok = np.array(window_arrays)

                # Menghitung mean dan median per piksel vertikal dengan mengabaikan matriks kosong
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    mean_blok = np.nanmean(stack_blok, axis=0)
                    median_blok = np.nanmedian(stack_blok, axis=0)

                # Menuliskan baris piksel hasil komputasi ke file keluaran secara bertahap
                dst_mean.write(mean_blok, 1, window=window)
                dst_median.write(median_blok, 1, window=window)

        print(f"Selesai! Hasil analisis {spi_name} telah disimpan.")

    finally:
        # Menutup seluruh berkas secara paksa untuk mengembalikan alokasi memori virtual
        for src in src_files:
            src.close()

print(f"\n{'='*60}")
print("Seluruh rangkaian kalkulasi statistik SPI (2010-2020) telah selesai dieksekusi dengan integritas data yang terverifikasi.")

Memulai agregasi spasial untuk SPI-1...
Ditemukan 132 file raster UNIK dalam rentang Jan 2010 - Des 2020.
Selesai! Hasil analisis SPI-1 telah disimpan.
Memulai agregasi spasial untuk SPI-3...
Ditemukan 132 file raster UNIK dalam rentang Jan 2010 - Des 2020.
Selesai! Hasil analisis SPI-3 telah disimpan.
Memulai agregasi spasial untuk SPI-6...
Ditemukan 132 file raster UNIK dalam rentang Jan 2010 - Des 2020.
Selesai! Hasil analisis SPI-6 telah disimpan.
Memulai agregasi spasial untuk SPI-12...
Ditemukan 132 file raster UNIK dalam rentang Jan 2010 - Des 2020.
Selesai! Hasil analisis SPI-12 telah disimpan.

Seluruh rangkaian kalkulasi statistik SPI (2010-2020) telah selesai dieksekusi dengan integritas data yang terverifikasi.


In [ ]:
# ==============================================================================
# TAHAP 4: KLASIFIKASI TINGKAT KEKERINGAN SPI (STANDAR BMKG/WMO)
# ==============================================================================
import os
import glob
import numpy as np
import rasterio

# 1. Definisikan direktori input dan output
# Menggunakan folder hasil komputasi Tahap 3 sebagai input
input_stats_folder = os.path.join(base_path, 'Statistik_SPI_Output_2010_2020')

# Membuat folder baru khusus untuk hasil klasifikasi
output_class_folder = os.path.join(base_path, 'Klasifikasi_SPI_2010_2020')
os.makedirs(output_class_folder, exist_ok=True)

# 2. Cari semua file raster mean dan median yang telah dihasilkan
stat_files = glob.glob(os.path.join(input_stats_folder, '*.tif'))
print(f"Ditemukan {len(stat_files)} file statistik SPI untuk diklasifikasi.\n")

for file_path in stat_files:
    filename = os.path.basename(file_path)

    # Memodifikasi nama file untuk output klasifikasi
    out_filename = filename.replace('.tif', '_class.tif')
    out_path = os.path.join(output_class_folder, out_filename)

    print(f"Melakukan reklasifikasi pada: {filename}...")

    with rasterio.open(file_path) as src:
        # Membaca seluruh data matriks
        data = src.read(1)
        meta = src.meta.copy()

        # 3. Menerapkan Logika Klasifikasi (Reclassify)
        # Menggunakan np.select untuk memetakan kondisi ke dalam nilai kelas (1-5)
        conditions = [
            (data <= -2.00),                         # Kelas 5: Sangat Kering
            (data > -2.00) & (data <= -1.50),        # Kelas 4: Kering
            (data > -1.50) & (data <= -1.00),        # Kelas 3: Agak Kering
            (data > -1.00) & (data <= 0.99),         # Kelas 2: Mendekati Normal
            (data > 0.99)                            # Kelas 1: Tidak Ada Kekeringan
        ]

        choices = [5, 4, 3, 2, 1]

        # Eksekusi klasifikasi, nilai yang tidak memenuhi syarat (misal area kosong) diisi 0
        class_data = np.select(conditions, choices, default=0)

        # 4. Penanganan Area Kosong (NoData)
        # Memastikan piksel lautan atau luar batas administrasi tetap bernilai 0 (NoData)
        class_data[np.isnan(data)] = 0

        # 5. Pembaruan Metadata
        # Mengubah tipe data raster dari Float32 menjadi Unsigned Integer 8-bit (uint8)
        # Hal ini sangat menghemat ukuran file karena nilainya hanya 0 sampai 5
        meta.update(
            dtype=rasterio.uint8,
            nodata=0
        )

        # 6. Menyimpan hasil klasifikasi
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(class_data, 1)

print(f"\n{'='*60}")
print(f"Proses klasifikasi selesai! Seluruh raster kategorikal tersimpan di:\n{output_class_folder}")

Ditemukan 8 file statistik SPI untuk diklasifikasi.

Melakukan reklasifikasi pada: SPI-1_mean_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-1_median_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-3_mean_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-3_median_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-6_mean_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-6_median_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-12_mean_2010_2020.tif...
Melakukan reklasifikasi pada: SPI-12_median_2010_2020.tif...

Proses klasifikasi selesai! Seluruh raster kategorikal tersimpan di:
/content/drive/MyDrive/Colab Notebooks - jariyanarifudin@mail.ugm.ac.id/Skripsi/Klasifikasi_SPI_2010_2020
